In [1]:
# Install libraries
!pip install pandas
%pip install nbconvert

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP

In [3]:
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the file path of data
invoice_export_path = os.path.join(notebook_dir, "data", "raw", "invoice_export_q1.xlsx")
print(invoice_export_path)

/Users/huien/Documents/opus-tuition/src/data_cleaning/data/raw/invoice_export_q1.xlsx


#### Read the dataset

In [4]:
# Read the dataset
df = pd.read_excel(invoice_export_path)

# Get first five row of the dataset
df.head()

,Invoice ID,Tutor ID,Student Name,Invoice Date,Amount,Payment Status,Payment Date,Notes
0,INV-2025-001,TAS-001,Lim Wei Jie,2025-01-31,SGD 165.00,Paid,2025-02-05,NaN
1,INV-2025-002,TAS-002,Tan Xiu Ying,2025-03-31,SGD 90.00,Pending,NaN,Awaiting bank transfer
2,INV-2025-003,TAS-003,Muhammad Hafiz,2025-02-28,SGD 240.00,Paid,2025-03-03,NaN
3,INV-2025-004,TAS-004,Chloe Wong,2025-03-31,SGD 75.00,PENDING,NaN,No response from parent
4,INV-2025-005,TAS-006,Ethan Ng,2025-04-30,SGD 70.00,pend,NaN,NaN


#### Analyse the dataset

In [5]:
# Get the number of columns and rows of the dataset
df.shape

(19, 8)

In [6]:
# Get the column names
column_names = list(df.columns)
print(column_names)

['Invoice ID', 'Tutor ID', 'Student Name', 'Invoice Date', 'Amount', 'Payment Status', 'Payment Date', 'Notes']


In [7]:
# View summary of dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Invoice ID      19 non-null     str  
 1   Tutor ID        19 non-null     str  
 2   Student Name    19 non-null     str  
 3   Invoice Date    19 non-null     str  
 4   Amount          19 non-null     str  
 5   Payment Status  19 non-null     str  
 6   Payment Date    13 non-null     str  
 7   Notes           4 non-null      str  
dtypes: str(8)
memory usage: 1.3 KB


In [8]:
# Get distinct value for payment status
payment_status = list(df["Payment Status"].unique())
print(payment_status)

['Paid', 'Pending', 'PENDING', 'pend', 'Pending ']


In [9]:
# Identify duplicated rows
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows)

      Invoice ID Tutor ID       Student Name Invoice Date      Amount  \
0   INV-2025-001  TAS-001        Lim Wei Jie   2025-01-31  SGD 165.00   
1   INV-2025-002  TAS-002       Tan Xiu Ying   2025-03-31   SGD 90.00   
2   INV-2025-003  TAS-003     Muhammad Hafiz   2025-02-28  SGD 240.00   
6   INV-2025-007  TAS-008  Aisha Binte Yusof   2025-07-31   SGD 90.00   
7   INV-2025-008  TAS-010         Darren Foo   2025-08-31  SGD 165.00   
8   INV-2025-001  TAS-001        Lim Wei Jie   2025-01-31  SGD 165.00   
10  INV-2025-003  TAS-003     Muhammad Hafiz   2025-02-28  SGD 240.00   
12  INV-2025-002  TAS-002       Tan Xiu Ying   2025-03-31   SGD 90.00   
14  INV-2025-007  TAS-008  Aisha Binte Yusof   2025-07-31   SGD 90.00   
16  INV-2025-008  TAS-010         Darren Foo   2025-08-31  SGD 165.00   
18  INV-2025-001  TAS-001        Lim Wei Jie   2025-01-31  SGD 165.00   

   Payment Status Payment Date                   Notes  
0            Paid   2025-02-05                     NaN  
1        

#### Clean dataset

In [10]:
# Remove duplicates, leaving only first occurrence
df_cleaned = df.drop_duplicates(keep="first")

In [11]:
def parse_to_date(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    expected_formats = [
        "%Y-%m-%d",     # 2025-03-12
        "%m/%d/%Y",     # 01/05/2025
        "%d-%m-%Y",     # 25-04-2025
        "%d %b %Y",     # 12 Feb 2025
        "%B %d, %Y",    # July 7, 2025
        "%d-%b-%Y",     # 15-Oct-2025
        "%d/%m/%y"      # 12/03/25
    ]

    parsed_values = []
    error_log = []

    # Loop only through the target column
    for idx, raw_val in df[col_name].items():
        if pd.isna(raw_val) or str(raw_val).strip() in ["", "None", "NaN"]:
            parsed_values.append(pd.NaT)
            continue

        clean_val = str(raw_val).strip()
        success = False

        for fmt in expected_formats:
            try:
                parsed_values.append(datetime.strptime(clean_val, fmt))
                success = True
                break
            except ValueError:
                continue

        if not success:
            # Capture specific row index and offending value
            error_log.append(f"Row {idx}: '{raw_val}' cannot be parsed.")
            parsed_values.append(pd.NaT)

    # If any row failed, stop everything and throw a specific error
    if error_log:
        raise ValueError(
            f"Data Integrity Error in column [{col_name}].\n"
            f"Details:\n" + "\n".join(error_log)
        )

    # If all passed, overwrite only that single column with the datetime type
    df[col_name] = parsed_values
    return df

df_cleaned = parse_to_date(df_cleaned, "Invoice Date")
df_cleaned = parse_to_date(df_cleaned, "Payment Date")

In [12]:
# Extract numbers, periods, and commas. Remove commas.
df_cleaned["Amount"] = (
    df_cleaned["Amount"]
    .astype(str)
    .str.extract(r"([\d.,]+)")[0]
    .str.replace(",", "", regex=False)
)

# Convert to Decimal and strictly round to 2 decimal places
def to_rounded_decimal(val):
    if pd.isna(val) or val == "nan":
        return None
    # Quantum ensures exactly two decimal places (.01)
    return Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

df_cleaned["Amount"] = df_cleaned["Amount"].apply(to_rounded_decimal)

In [13]:
# Strip leading, trailing, and internal excess whitespace
df_cleaned["Payment Status"] = (df_cleaned["Payment Status"].str.strip().str.replace(r"\s+", " ", regex=True))

# Convert text into lowercase then capitalise the first letter of the word
df_cleaned["Payment Status"] = df_cleaned["Payment Status"].str.lower().str.capitalize()

# Replace "Pend" with "Pending"
df_cleaned["Payment Status"] = df_cleaned["Payment Status"].replace({"Pend": "Pending"})

In [15]:
# Convert selected columns to the modern string dtype
df_cleaned = df_cleaned.astype({
    "Invoice ID": "string",
    "Tutor ID": "string",
    "Student Name": "string",
    "Payment Status": "string",
    "Notes": "string"
})

#### Validate dataset

In [14]:
# Check that payment status now only have 2 values
print(list(df_cleaned["Payment Status"].unique()))

['Paid', 'Pending']


In [16]:
# Check the final data type
df_cleaned.dtypes

Invoice ID                string
Tutor ID                  string
Student Name              string
Invoice Date      datetime64[us]
Amount                    object
Payment Status            string
Payment Date      datetime64[us]
Notes                     string
dtype: object

In [17]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 13 entries, 0 to 17
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Invoice ID      13 non-null     string        
 1   Tutor ID        13 non-null     string        
 2   Student Name    13 non-null     string        
 3   Invoice Date    13 non-null     datetime64[us]
 4   Amount          13 non-null     object        
 5   Payment Status  13 non-null     string        
 6   Payment Date    8 non-null      datetime64[us]
 7   Notes           3 non-null      string        
dtypes: datetime64[us](2), object(1), string(5)
memory usage: 936.0+ bytes


In [18]:
# Print the final cleaned results
df_cleaned

,Invoice ID,Tutor ID,Student Name,Invoice Date,Amount,Payment Status,Payment Date,Notes
0,INV-2025-001,TAS-001,Lim Wei Jie,2025-01-31,165.00,Paid,2025-02-05,<NA>
1,INV-2025-002,TAS-002,Tan Xiu Ying,2025-03-31,90.00,Pending,NaT,Awaiting bank transfer
2,INV-2025-003,TAS-003,Muhammad Hafiz,2025-02-28,240.00,Paid,2025-03-03,<NA>
3,INV-2025-004,TAS-004,Chloe Wong,2025-03-31,75.00,Pending,NaT,No response from parent
4,INV-2025-005,TAS-006,Ethan Ng,2025-04-30,70.00,Pending,NaT,<NA>
5,INV-2025-006,TAS-007,Hafiz Bin Aris,2025-06-30,112.50,Pending,NaT,Note trailing space
6,INV-2025-007,TAS-008,Aisha Binte Yusof,2025-07-31,90.00,Paid,2025-08-02,<NA>
7,INV-2025-008,TAS-010,Darren Foo,2025-08-31,165.00,Paid,2025-09-01,<NA>
9,INV-2025-009,TAS-011,Raeanne Ong,2025-09-30,120.00,Paid,2025-10-04,<NA>
11,INV-2025-010,TAS-012,Jordan Tan,2025-10-31,135.00,Pending,NaT,<NA>
